# **Processamento de Linguagem Natural [2025-Q3]**
Prof. Alexandre Donizeti Alves

### **PROJETO PRÁTICO** [LangChain + Grandes Modelos de Linguagem]


O **PROJETO PRÁTICO** deve ser feito utilizando o **Google Colab** com uma conta sua vinculada ao Gmail. O link do seu notebook armazenado no Google Drive e o link de um repositório no GitHub devem ser enviados usando o seguinte formulário:

> https://forms.gle/D4gLqP1iGgyn2hbH8


**IMPORTANTE**: A submissão deve ser feita até o dia **07/12 (domingo)** APENAS POR UM INTEGRANTE DA EQUIPE, até às 23h59. Por favor, lembre-se de dar permissão de ACESSO IRRESTRITO para o professor da disciplina.

### **EQUIPE**

---

**POR FAVOR, PREENCHER OS INTEGRANDES DA SUA EQUIPE:**


**Integrante 01:**

`Gabriel Victor Lima Gonçalves RA: 11202230490`

**Integrante 02:**

`Por favor, informe o seu nome completo e RA:`

**Integrante 03:**

`Por favor, informe o seu nome completo e RA:`

### **GRANDE MODELO DE LINGUAGEM (*Large Language Model - LLM*)**

---

Cada equipe deve selecionar um Grande Modelo de Linguagem (*Large Language Model - LMM*).



Por favor, informe os dados do LLM selecionada:

>


**LLM**:

> chatgpt-5-nano OpenAI

**Link para a documentação oficial**:

> https://platform.openai.com/docs/api-reference/introduction

### **API (Opcional)**
---

Por favor, informe os dados da API selecionada:

**API**:

**Site oficial**:

**Link para a documentação oficial**:






### **DESCRIÇÃO**
---

Implementar um `notebook` no `Google Colab` que faça uso do framework **`LangChain`** (obrigatório) e de um **LLM** aplicando, no mínimo, DUAS técnicas de PLN. As técnicas podem ser aplicada em qualquer córpus obtido a partir de uma **API** ou a partir de uma página Web.

O **LLM** e a **API** selecionados devem ser informados na seguinte planilha:

> https://docs.google.com/spreadsheets/d/1iIUZcwnywO7RuF6VEJ8Rx9NDT1cwteyvsnkhYr0NWtU/edit?usp=sharing

>
As seguintes técnicas de PLN podem ser usadas:

*   Correção Gramatical
*   Classificação de Textos
*   Análise de Sentimentos
*   Detecção de Emoções
*   Extração de Palavras-chave
*   Tradução de Textos
*   Sumarização de Textos
*   Similaridade de Textos
*   Reconhecimento de Entidades Nomeadas
*   Sistemas de Perguntas e Respostas
>

**IMPORTANTE:** É obrigatório usar o e-mail da UFABC.


### **CRITÉRIOS DE AVALIAÇÃO**
---


Serão considerados como critérios de avaliação os seguintes pontos:

* Uso do framework **`LangChain`**.

* Escolha e uso de um **LLM**.

* Escolha e uso de uma **API** ou **Página Web**.

* Projeto disponível no Github.

* Apresentação (5 a 10 minutos).

* Criatividade no uso do framework **`LangChain`** em conjunto com o **LLM** e a **API**.




**IMPORTANTE**: todo o código do notebook deve ser executado. Código sem execução não será considerado.

### **IMPLEMENTAÇÃO**
---

# Final

In [10]:
# por favor, inserir o código a partir daqui...




# Estruturação

## Pipeline Recebendo os dados da API e tratando 

## Pipeline Final Resumida

### Request da API filtrando areas tech

In [3]:
#Imports
import os
import re
import json
import requests
from pathlib import Path

In [4]:
import os
import json
import re
import requests


def pipeline_remotive_tech_jobs():
    """
    Pipeline completo:
    - Faz o request para a API da Remotive
    - Filtra SOMENTE vagas tech
    - Salva em ../data/raw/tech_jobs.json
    """

    # Categorias tech
    TECH_CATEGORIES = [
        "software-development",
        "ai-ml",
        "data",
        "devops",
        "qa"
    ]

    def slugify_category(text: str) -> str:
        return re.sub(r"\s+", "-", text.strip().lower())

    url = "https://remotive.com/api/remote-jobs"

    # ---- REQUEST ----
    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()
    except Exception as e:
        print(f"❌ Erro ao fazer request: {e}")
        return

    data = response.json()
    raw_jobs = data.get("jobs", [])

    # ---- FILTRAGEM ----
    tech_jobs = []
    for job in raw_jobs:
        category_raw = job.get("category", "")
        category_slug = slugify_category(category_raw)

        if category_slug in TECH_CATEGORIES:
            tech_jobs.append(job)

    # ---- SAVE JSON ----
    save_path = os.path.join("..", "data", "raw", "tech_jobs.json")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(tech_jobs, f, ensure_ascii=False, indent=4)

    print(f"✔ Arquivo salvo em: {save_path}")
    print(f"✔ Salvo {len(tech_jobs)} tech jobs em tech_jobs.json")


#pipeline_remotive_tech_jobs()

### Recebe dados da API e faz tratamento inicial

In [5]:

# =========================================================
# 1. CLEAN HTML
# =========================================================
def clean_html(text: str) -> str:
    """Remove todas as tags HTML."""
    return re.sub(r"<[^>]+>", "", text or "")


# =========================================================
# 2. CLEAN JOB LIST (caso use o retorno da API diretamente)
# =========================================================
def clean_jobs(raw_json: dict) -> list:
    """
    Recebe o JSON da API e retorna uma nova lista de jobs com description limpa.
    """
    raw_jobs = raw_json["jobs"]
    cleaned = []

    for job in raw_jobs:
        new_job = job.copy()
        new_job["description"] = clean_html(job.get("description", ""))
        cleaned.append(new_job)

    return cleaned


# =========================================================
# 3. LOAD SUPPORT MAPS (tech + soft)
# =========================================================
def load_support_maps(support_dir: Path):
    with open(support_dir / "tech_stack.json", "r", encoding="utf-8") as f:
        tech_map = json.load(f)

    with open(support_dir / "soft_skills.json", "r", encoding="utf-8") as f:
        soft_map = json.load(f)

    # Normalize tudo para lowercase
    tech_map = {k.lower(): [v.lower() for v in values] for k, values in tech_map.items()}
    soft_map = {k.lower(): [v.lower() for v in values] for k, values in soft_map.items()}

    return tech_map, soft_map


# =========================================================
# 4. EXTRACT SKILL (macro + variações usadas)
#     → AGORA USANDO REGEX QUE GARANTE PALAVRA INTEIRA
#     → NÃO TEM MAIS FALSOS POSITIVOS (go, r, c...)
# =========================================================
def extract_skill_map(description: str, skill_map: dict):
    desc = description.lower()
    found = {}

    for macro, variations in skill_map.items():
        matches = []

        for var in variations:
            # Regex para casar palavra/frase inteira
            pattern = r"\b" + re.escape(var.lower()) + r"\b"

            if re.search(pattern, desc):
                matches.append(var)
                # Remove para evitar duplicações
                desc = re.sub(pattern, " ", desc)

        if matches:
            found[macro] = sorted(set(matches))  # organiza e remove duplicadas

    # Limpa excessos de espaços após substituições
    desc = re.sub(r"\s+", " ", desc).strip()

    return desc, found


# =========================================================
# 5. PROCESS A SINGLE JOB
# =========================================================
def process_job(job, tech_map, soft_map):
    desc = clean_html(job.get("description", "")).lower()

    # tech
    desc, tech_matches = extract_skill_map(desc, tech_map)

    # soft
    desc, soft_matches = extract_skill_map(desc, soft_map)

    job["description"] = desc
    job["tech_stacks_found"] = tech_matches
    job["soft_skills_found"] = soft_matches

    job.pop("tags", None)

    return job


# =========================================================
# 6. MAIN PIPELINE
# =========================================================
def pipeline(
    raw_path: Path = Path("../data/raw/tech_jobs.json"),
    processed_path: Path = Path("../data/processed/jobs_processed.json"),
    support_dir: Path = Path("../support")
):
    # Load tech+soft reference maps
    tech_map, soft_map = load_support_maps(support_dir)

    # Load raw jobs
    with open(raw_path, "r", encoding="utf-8") as f:
        raw_jobs = json.load(f)

    # Process everything
    processed = [process_job(job, tech_map, soft_map) for job in raw_jobs]

    # Ensure output directory exists
    processed_path.parent.mkdir(parents=True, exist_ok=True)

    # Save JSON
    with open(processed_path, "w", encoding="utf-8") as f:
        json.dump(processed, f, indent=4, ensure_ascii=False)

    print(f"✔ Arquivo salvo em: {processed_path}")


# =========================================================
# RUN DIRECTLY
# =========================================================

## pipeline trata dados diretamente do arquivo ../data/raw/tech_jobs.json bruto recebido após request da api e salva em ../data/processed/jobs_processed.json
#pipeline()

### Embedding Pipe

In [6]:
import pandas
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()  # procura automaticamente um .env ou .config no projeto
key = os.getenv("openai")
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
    api_key= key
    )

In [7]:
def generate_embeddings_df(
        
    path_json="../data/processed/jobs_processed.json",
    output_csv="../data/processed/jobs_with_embeddings.csv",
    model_name="text-embedding-3-large",
):
    
    """
    Carrega o JSON processado, trata colunas, gera embeddings e salva CSV final.
    Retorna o dataframe resultante.
    """

    # 1. Carregar dataframe
    df = pd.read_json(path_json)

    # 2. Colunas que precisam ser convertidas de str -> list
    cols_list = ["tech_stacks_found", "soft_skills_found"]

    for col in cols_list:
        df[col] = df[col].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )

    # 4. Gerar embeddings
    df["embedding"] = df["description"].apply(embeddings.embed_query)

    # 5. Salvar CSV final
    df.to_csv(output_csv, index=False)

    print(f"✅ CSV salvo em: {output_csv}")

    return df

### Nota currículo

In [8]:
curriculo = """GABRIEL VICTOR LIMA GONC¸ ALVES
+55 (11)94924-4811 ⋄ Santo Andr´e, SP
gabrielvgonc@gmail.com ⋄ www.linkedin.com/in/gabriel-victor-71187b223
OBJECTIVE
Seeking career growth opportunities in the field of software development.
EDUCATION
Bachelor of Computer Science, Universidade Federal do ABC Expected Graduation: 2026
SKILLS
Main Technical Skills Python, SQL, ETL, Java, Git
Soft Skills Communication, Adaptability, Analytical Thinking, Proactivity
Github Portfolio https://github.com/Netreck
Languages English, Portuguese
PROFESSIONAL EXPERIENCE
Intern – Bank of America Jun 2025 – Present
VP Global Technology – Tech Rotation Program
• Developed test automation and tools to support QA teams in payment systems.
• Contributed to internal automation and process optimization projects.
• Technologies: Java, Python, Git, SQL, Node.js, HTML, CSS, Octane, QTest, Matera.
Data Intern – Vivo (Telefˆonica Brasil) Jan 2025 – Jun 2025
VP Engineering & Customer Services
• Generated, monitored, and analyzed operational KPIs focused on efficiency and customer experience.
• Built Machine Learning solutions to modernize and improve operational results.
• Technologies: Python, SQL, TensorFlow, Power BI, Excel.
EXTRACURRICULAR ACTIVITIES
Member – Green Team Hacker Club Jan 2024 – Jan 2025
Project Manager – Data Projects, Green Team Hacker Club Jan 2025 – Present
Federal University of ABC (UFABC) Santo Andr´e, SP
• Official student-led organization linked to UFABC.
• Participated in classes and projects related to Data Science.
• Defined and managed technology stack for projects, considering scalability, performance, and integration.
• Managed team activities including task delegation, progress tracking, and technical support.
• Worked with ETL, process automation (pipelines), SQL, PostgreSQL, Data Visualization (Seaborn,
Matplotlib), Data Modeling, Machine Learning models (Scikit-learn, TensorFlow), model deployment
via APIs, MLflow, and LLMs.
"""

In [9]:
def run_matching_pipeline(curriculo_str):

    # ================================================
    # IMPORTS
    # ================================================
    import json, re, ast, os
    import pandas as pd
    import numpy as np
    from pathlib import Path
    from dotenv import load_dotenv
    from sklearn.metrics.pairwise import cosine_similarity
    from langchain_openai import OpenAIEmbeddings


    # ================================================
    # SUPPORT MAPS
    # ================================================
    def load_support_maps(support_dir: Path):
        with open(support_dir / "tech_stack.json", "r", encoding="utf-8") as f:
            tech_map = json.load(f)

        with open(support_dir / "soft_skills.json", "r", encoding="utf-8") as f:
            soft_map = json.load(f)

        tech_map = {k.lower(): [v.lower() for v in values] for k, values in tech_map.items()}
        soft_map = {k.lower(): [v.lower() for v in values] for k, values in soft_map.items()}

        return tech_map, soft_map


    # ================================================
    # EXTRACT SKILL
    # ================================================
    def extract_skill_map(description: str, skill_map: dict):
        desc = description.lower()
        found = {}

        for macro, variations in skill_map.items():
            matches = []
            for var in variations:
                if var in desc:
                    matches.append(var)
                    desc = desc.replace(var, "")

            if matches:
                found[macro] = matches

        desc = re.sub(r"\s+", " ", desc).strip()
        return desc, found


    # ================================================
    # PROCESS CURRICULO
    # ================================================
    def process_curriculo(curriculo: str, support_dir: Path = Path("../support")):

        load_dotenv()
        api_key = os.getenv("openai")
        if not api_key:
            raise ValueError("❌ OPENAI_API_KEY não encontrada no .env")

        tech_map, soft_map = load_support_maps(support_dir)

        clean_curriculo = re.sub(r"\s+", " ", curriculo).strip()

        desc_after_tech, tech_found = extract_skill_map(clean_curriculo, tech_map)
        desc_after_soft, soft_found = extract_skill_map(desc_after_tech, soft_map)

        embeddings = OpenAIEmbeddings(
            model="text-embedding-3-large",
            api_key=api_key
        )

        embedding_vector = embeddings.embed_query(clean_curriculo)

        return {
            "curriculo_raw": curriculo,
            "curriculo_clean": clean_curriculo,
            "embedding": embedding_vector,
            "tech_stacks_found": tech_found,
            "soft_skills_found": soft_found
        }


    # ================================================
    # FINAL SCORE
    # ================================================
    def compute_final_score(similarity_raw, sim_min, sim_max,
                            job_techs, job_soft,
                            curr_techs, curr_soft):

        similarity_norm = (similarity_raw - sim_min) / (sim_max - sim_min + 1e-9)
        similarity_norm = max(0, min(1, similarity_norm))

        # Tech score
        total_tech = len(job_techs)
        tech_val = (
            sum(1 for macro in job_techs if macro in curr_techs) / total_tech
            if total_tech > 0 else 1
        )

        # Soft score
        total_soft = len(job_soft)
        soft_val = (
            sum(1 for macro in job_soft if macro in curr_soft) / total_soft
            if total_soft > 0 else 1
        )

        final = (
            0.8 * similarity_norm +
            0.1 * tech_val +
            0.1 * soft_val
        ) * 100

        return round(final, 2)


    # ================================================
    # PRINT BONITO
    # ================================================
    def print_job_match(row, index):
        print("======================================================================")
        print(f"📌 Job #{index}")
        print(f"🏷️  Title: {row['title']}")
        print(f"🏢 Company: {row['company_name']}")
        print(f"🔧 Tech Stacks Found: {row['tech_stacks_found']}")
        print(f"📊 Similarity: {round(row['embedding_similarity'], 4)}")
        print(f"🔥 Compatibility Score: {row['compatibility_score']}")
        print("----------------------------------------------------------------------")
        print("📝 Description:")
        desc = row['description']
        if len(desc) > 500:
            desc = desc[:500] + "..."
        print(desc)
        print("======================================================================\n\n")


    # ================================================
    # EXECUÇÃO FINAL
    # ================================================

    # 1. Processa currículo
    resultado = process_curriculo(curriculo_str)

    # 2. Carrega CSV
    df = pd.read_csv("../data/processed/jobs_with_embeddings.csv")

    # 3. Converte strings → dict/list
    cols_to_fix = ["tech_stacks_found", "soft_skills_found", "embedding"]
    for col in cols_to_fix:
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    # 4. Similaridade
    curr_emb = np.array(resultado["embedding"]).reshape(1, -1)

    df["embedding_similarity"] = df["embedding"].apply(
        lambda v: float(
            cosine_similarity(curr_emb, np.array(v).reshape(1, -1))[0][0]
        )
    )

    # 5. Score final
    sim_min = df["embedding_similarity"].min()
    sim_max = df["embedding_similarity"].max()

    df["compatibility_score"] = df.apply(lambda row:
        compute_final_score(
            similarity_raw=row["embedding_similarity"],
            sim_min=sim_min,
            sim_max=sim_max,
            job_techs=row["tech_stacks_found"],
            job_soft=row["soft_skills_found"],
            curr_techs=resultado["tech_stacks_found"],
            curr_soft=resultado["soft_skills_found"]
        ),
        axis=1
    )

    # 6. Ordena
    df_sorted = df.sort_values(by="compatibility_score", ascending=False)

    # 7. Print bonito (Top 10)
    for i, row in df_sorted.head(100).iterrows():
        print_job_match(row, i)

In [10]:
import json
import re
from pathlib import Path


# =========================================================
# 1. LOAD SUPPORT MAPS (tech + soft)
# =========================================================
def load_support_maps(support_dir: Path):
    with open(support_dir / "tech_stack.json", "r", encoding="utf-8") as f:
        tech_map = json.load(f)

    with open(support_dir / "soft_skills.json", "r", encoding="utf-8") as f:
        soft_map = json.load(f)

    tech_map = {k.lower(): [v.lower() for v in values] for k, values in tech_map.items()}
    soft_map = {k.lower(): [v.lower() for v in values] for k, values in soft_map.items()}

    return tech_map, soft_map


# =========================================================
# 2. **EXTRACT SKILL CORRIGIDO** (match por palavra inteira)
# =========================================================
def extract_skill_map(description: str, skill_map: dict):
    desc = description.lower()
    found = {}

    for macro, variations in skill_map.items():
        matches = []

        for var in variations:
            # Regex para casar palavra inteira — evita falsos positivos
            pattern = r"\b" + re.escape(var.lower()) + r"\b"

            if re.search(pattern, desc):
                matches.append(var)
                # Remove a ocorrência encontrada para evitar duplicação
                desc = re.sub(pattern, " ", desc)

        if matches:
            found[macro] = sorted(set(matches))

    desc = re.sub(r"\s+", " ", desc).strip()
    return desc, found


# =========================================================
# 3. PIPELINE DO CURRÍCULO (SEM EMBEDDING)
# =========================================================
def process_curriculo(
    curriculo: str,
    support_dir: Path = Path("../support")
):
    tech_map, soft_map = load_support_maps(support_dir)

    clean_curriculo = re.sub(r"\s+", " ", curriculo).strip()

    # Extrai tech
    desc_after_tech, tech_found = extract_skill_map(clean_curriculo, tech_map)
    # Extrai soft
    desc_after_soft, soft_found = extract_skill_map(desc_after_tech, soft_map)

    # Mesma estrutura de uma vaga
    return {
        "description": clean_curriculo,
        "tech_stacks_found": tech_found,
        "soft_skills_found": soft_found
    }


# =========================================================
# 4. PRINT BONITO — MESMO FORMATO DAS VAGAS
# =========================================================
def print_curriculo_formatado(resultado):
    print("======================================================================")
    print("📌 CURRÍCULO PROCESSADO")
    print("🔧 Tech Stacks Found:", resultado["tech_stacks_found"])
    print("🧩 Soft Skills Found:", resultado["soft_skills_found"])
    print("----------------------------------------------------------------------")
    print("📝 Description:")

    desc = resultado["description"]
    if len(desc) > 700:
        desc = desc[:700] + "..."

    print(desc)
    print("======================================================================\n")


# =========================================================
# 5. FUNÇÃO FINAL
# =========================================================
def run_curriculo_preview(curriculo_str):
    resultado = process_curriculo(curriculo_str)
    print_curriculo_formatado(resultado)
    return resultado

In [11]:
run_curriculo_preview(curriculo)

📌 CURRÍCULO PROCESSADO
🔧 Tech Stacks Found: {'languages': ['java', 'python'], 'backend_frameworks': ['node'], 'databases_sql': ['postgresql'], 'ml_ai': ['scikit-learn', 'tensorflow'], 'bi_analytics': ['power bi'], 'others_general': ['automation', 'computer science', 'css', 'data science', 'data visualization', 'etl', 'excel', 'git', 'github', 'html', 'sql', 'technical support']}
🧩 Soft Skills Found: {'communication': ['communication'], 'leadership': ['delegation'], 'problem_solving': ['analytical thinking'], 'adaptability': ['adaptability'], 'time_management': ['organization'], 'work_style': ['proactivity'], 'others_general': ['soft skills']}
----------------------------------------------------------------------
📝 Description:
GABRIEL VICTOR LIMA GONC¸ ALVES +55 (11)94924-4811 ⋄ Santo Andr´e, SP gabrielvgonc@gmail.com ⋄ www.linkedin.com/in/gabriel-victor-71187b223 OBJECTIVE Seeking career growth opportunities in the field of software development. EDUCATION Bachelor of Computer Science,

{'description': 'GABRIEL VICTOR LIMA GONC¸ ALVES +55 (11)94924-4811 ⋄ Santo Andr´e, SP gabrielvgonc@gmail.com ⋄ www.linkedin.com/in/gabriel-victor-71187b223 OBJECTIVE Seeking career growth opportunities in the field of software development. EDUCATION Bachelor of Computer Science, Universidade Federal do ABC Expected Graduation: 2026 SKILLS Main Technical Skills Python, SQL, ETL, Java, Git Soft Skills Communication, Adaptability, Analytical Thinking, Proactivity Github Portfolio https://github.com/Netreck Languages English, Portuguese PROFESSIONAL EXPERIENCE Intern – Bank of America Jun 2025 – Present VP Global Technology – Tech Rotation Program • Developed test automation and tools to support QA teams in payment systems. • Contributed to internal automation and process optimization projects. • Technologies: Java, Python, Git, SQL, Node.js, HTML, CSS, Octane, QTest, Matera. Data Intern – Vivo (Telefˆonica Brasil) Jan 2025 – Jun 2025 VP Engineering & Customer Services • Generated, monito

In [15]:
#resultado = process_curriculo(curriculo)
run_matching_pipeline(curriculo)

📌 Job #103
🏷️  Title: Software Developer - Intern
🏢 Company: SYSTEM AUTOMATION CORPORATION
🔧 Tech Stacks Found: {'languages': ['java', 'javascript'], 'others_general': ['automation', 'cloud', 'computer science', 'documentation', 'product development', 'saas', 'solid', 'sql', 'testing']}
📊 Similarity: 0.4037
🔥 Compatibility Score: 88.62
----------------------------------------------------------------------
📝 Description:
apply job type internship description internship summer period -early december through mid january 2026about the company system (sa) is a software vendor focused on providing systems that support government regulatory management operations.founded in the district of columbia in 1968 and originally contracted to develop and support the us army recruiting system (contract held for 42 years), sa has since become an industry leader in designing, developing, implementing, and maintaining comprehensiv...


📌 Job #135
🏷️  Title: Python Developer
🏢 Company: Hotelogix India Priv

In [14]:
#resultado = process_curriculo(curriculo)
run_matching_pipeline(curriculo)

📌 Job #103
🏷️  Title: Software Developer - Intern
🏢 Company: SYSTEM AUTOMATION CORPORATION
🔧 Tech Stacks Found: {'languages': ['java', 'javascript'], 'others_general': ['automation', 'cloud', 'computer science', 'documentation', 'product development', 'saas', 'solid', 'sql', 'testing']}
📊 Similarity: 0.4036
🔥 Compatibility Score: 88.82
----------------------------------------------------------------------
📝 Description:
apply job type internship description internship summer period -early december through mid january 2026about the company system (sa) is a software vendor focused on providing systems that support government regulatory management operations.founded in the district of columbia in 1968 and originally contracted to develop and support the us army recruiting system (contract held for 42 years), sa has since become an industry leader in designing, developing, implementing, and maintaining comprehensiv...


📌 Job #135
🏷️  Title: Python Developer
🏢 Company: Hotelogix India Priv

### Rodando Pipeline

In [ ]:
# Pipe- gera_dados
pipeline_remotive_tech_jobs() #faz request e salva em data/raw
pipeline() # pega os dados salvos em data/raw trata eles e salva em processed com as tech_stacks, soft_skills e description tratados
generate_embeddings_df() # Utiliza API openIA pra gerar embeddings para cada vaga, 


✔ Arquivo salvo em: ../data/raw/tech_jobs.json
✔ Salvo 347 tech jobs em tech_jobs.json
✔ Arquivo salvo em: ../data/processed/jobs_processed.json
✅ CSV salvo em: ../data/processed/jobs_with_embeddings.csv


,id,url,title,company_name,company_logo,category,job_type,publication_date,candidate_required_location,salary,description,tech_stacks_found,soft_skills_found,company_logo_url,embedding
0,2079713,https://remotive.com/remote-jobs/qa/qa-enginee...,QA Engineer,PatientNow,https://remotive.com/job/2079713/logo,QA,full_time,2025-11-21T20:50:23,USA,,"the roleas our founding qa engineer, you’ll bu...","{'frontend_frameworks': ['next.js'], 'testing'...","{'communication': ['communication'], 'attentio...",NaN,"[-0.028529442846775055, 0.008444409817457199, ..."
1,2070452,https://remotive.com/remote-jobs/software-deve...,Quantitative Research Team Lead (Completed),Apexver,https://remotive.com/job/2070452/logo,Software Development,full_time,2025-11-21T20:00:41,Worldwide,$180k + performance bonus,"role overview as the quantitative team lead, y...","{'languages': ['c', 'python'], 'others_general...","{'communication': ['communication'], 'teamwork...",NaN,"[-0.04142270237207413, 0.0007170444587245584, ..."
2,2079702,https://remotive.com/remote-jobs/qa/sr-perform...,Sr. Performance Tester,Tietoevry,https://remotive.com/job/2079702/logo,QA,full_time,2025-11-21T18:51:44,India,,company descriptionwe are developers of digita...,"{'languages': ['java', 'python'], 'cloud': ['a...","{'communication': ['communication'], 'creativi...",NaN,"[-0.04076246917247772, -0.00853677000850439, -..."
3,2080480,https://remotive.com/remote-jobs/software-deve...,Senior Software Engineer - Platform Development,Tanium,https://remotive.com/job/2080480/logo,Software Development,full_time,2025-11-21T16:50:18,Canada,"c$200,000 to c$220,000",the basics: as a tanium senior software engine...,"{'languages': ['c', 'go'], 'cloud': ['aws', 'a...","{'teamwork': ['collaboration'], 'time_manageme...",NaN,"[-0.03667346388101578, -0.03649427741765976, -..."
4,2080479,https://remotive.com/remote-jobs/software-deve...,Staff Application Security Engineer,PlayStation Global,https://remotive.com/job/2080479/logo,Software Development,full_time,2025-11-21T16:50:17,USA,"$198,200 - $297,400 usd",why playstation? playstation isn’t just the be...,"{'languages': ['c', 'java', 'javascript', 'pyt...","{'communication': ['communication'], 'teamwork...",NaN,"[-0.010802310891449451, -0.02232527732849121, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
342,2054061,https://remotive.com/remote-jobs/software-deve...,Full Stack Developer,Getwingapp,https://remotive.com/job/2054061/logo,Software Development,full_time,2025-08-31T00:51:25,India,inr 15 lpa - 25 lpa,about uswing is seeking elite talent to join m...,"{'languages': ['go', 'javascript', 'php', 'pyt...","{'leadership': ['leadership'], 'work_style': [...",NaN,"[-0.03661661967635155, -0.016772883012890816, ..."
343,2053123,https://remotive.com/remote-jobs/qa/software-d...,Software Development Engineer in Test,"Bayesian Health, Inc.",https://remotive.com/job/2053123/logo,QA,full_time,2025-08-27T22:51:54,USA,,software development engineer in testin briefw...,"{'testing': ['cypress', 'selenium'], 'producti...","{'teamwork': ['collaboration'], 'problem_solvi...",NaN,"[0.0002656708238646388, 0.02247590944170952, -..."
344,2052625,https://remotive.com/remote-jobs/software-deve...,Internal Tooling Engineer Lead,Protex%20AI,https://remotive.com/job/2052625/logo,Software Development,full_time,2025-08-27T22:50:54,Hungary,,"about us:at protex ai, we are at the forefront...","{'languages': ['go', 'javascript', 'python'], ...","{'communication': ['communication'], 'accounta...",NaN,"[-0.025792598724365234, 0.000583440123591572, ..."
345,2052631,https://remotive.com/remote-jobs/software-deve...,Senior Fullstack Engineer,Ciklum,https://remotive.com/job/2052631/logo,Software Development,full_time,2025-08-27T22:50:53,Poland,,ciklum is looking for a senior engineer to joi...,"{'languages': ['java'], 'backend_frameworks': ...","{'teamwork': ['collaboration'], 'adaptability'...",NaN,"[-0.033299144357442856, -0.04704069346189499, ..."


## -------------------------------------------------------------------------

## Recebendo API e criando pipeline

In [11]:
### REQ Recebe currículos da API ### 

import requests
import json

url = "https://remotive.com/api/remote-jobs?limit=1"

response = requests.get(url)
response.raise_for_status()

raw_one_job = response.json()["jobs"][0]  # full raw dict for the job

print(json.dumps(raw_one_job, indent=2))

{
  "id": 1987878,
  "url": "https://remotive.com/remote-jobs/education/language-teachers-1987878",
  "title": "Language teachers",
  "company_name": "AE Virtual Class S.A",
  "company_logo": "https://remotive.com/job/1987878/logo",
  "category": "Education",
  "tags": [
    "teaching"
  ],
  "job_type": "part_time",
  "publication_date": "2025-11-21T05:30:55",
  "candidate_required_location": "Americas",
  "salary": "$10k",
  "description": "<p style=\"margin-bottom: 10px; caret-color: #6a6a6a; color: #6a6a6a;\"><strong>Description:</strong></p>\n<p style=\"margin-bottom: 10px; caret-color: #6a6a6a; color: #6a6a6a;\">AE Virtual Class, member of Academia Europea Group, leader in language teaching, with 56 years of experience and the largest teaching staff in the Americas! We are looking for language enthusiasts who want to be part of our great family! Experience is NOT a requirement! We teach you how to teach!</p>\n<p style=\"margin-bottom: 10px; caret-color: #6a6a6a; color: #6a6a6a;\"

### Limpando tags HTML da descrição

In [ ]:
import re

def clean_html(text: str) -> str:
    """Remove todas as tags HTML."""
    return re.sub(r"<[^>]+>", "", text)


def clean_jobs(raw_json: dict) -> list:
    """
    Recebe o JSON da API e retorna uma nova lista de jobs
    com "description" limpa.
    """
    raw_jobs = raw_json["jobs"]
    clean_jobs = []

    for job in raw_jobs:
        new_job = job.copy()
        new_job["description"] = clean_html(job["description"])
        clean_jobs.append(new_job)

    return clean_jobs


# ----------------------------- 
clean_list = clean_jobs(response.json())
print(clean_list[0]["description"])

Description:
AE Virtual Class, member of Academia Europea Group, leader in language teaching, with 56 years of experience and the largest teaching staff in the Americas! We are looking for language enthusiasts who want to be part of our great family! Experience is NOT a requirement! We teach you how to teach!
 
Job Requirements:
Laptop (with webcam).
Stable internet connection (15 Mbps).
Attitude.
Dynamism.
Afternoon and/or evening shifts (Central America time zone).
Excellent Mandarin, German, Italian or French proficiency (C1-C2).
 
Main responsibilities of the position:
Motivate students.
Generate interest in cultures and languages.
Prepare reports.
Evaluations.



### Request Aprimorada Apenas com vagas Tech

In [15]:
import requests
import json
import os

# Categorias realmente tech
TECH_CATEGORIES = [
    "software-development",
    "ai-ml",
    "data",
    "devops",
    "qa"
]

def get_remotive_tech_jobs():
    url = "https://remotive.com/api/remote-jobs"
    response = requests.get(url)
    data = response.json()

    raw_jobs = data.get("jobs", [])

    tech_jobs = []

    for job in raw_jobs:
        category_slug = job.get("category", "").lower().replace(" ", "-")

        if category_slug in TECH_CATEGORIES:
            tech_jobs.append(job)
    return tech_jobs


def save_to_json(data, filename="tech_jobs.json"):
    # Caminho final: ../data/raw/tech_jobs.json
    save_path = os.path.join("..", "data", "raw", filename)

    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"Arquivo salvo em: {save_path}")


tech_jobs = get_remotive_tech_jobs()
save_to_json(tech_jobs)

print(f"Salvo {len(tech_jobs)} tech jobs em tech_jobs.json")

Arquivo salvo em: ../data/raw/tech_jobs.json
Salvo 346 tech jobs em tech_jobs.json


### Retirada de stacks e softskills

In [ ]:
import json
import re
from pathlib import Path


# --------------------------
# 1. Caminhos
# --------------------------
RAW_PATH = Path("../data/raw/tech_jobs.json")
PROCESSED_PATH = Path("../data/processed/jobs_processed.json")
SUPPORT_DIR = Path("../support")


# --------------------------
# 3. Carregar tech/soft maps
# --------------------------
def load_support_maps():
    with open(SUPPORT_DIR / "tech_stack.json", "r", encoding="utf-8") as f:
        tech_map = json.load(f)

    with open(SUPPORT_DIR / "soft_skills.json", "r", encoding="utf-8") as f:
        soft_map = json.load(f)

    tech_map = {k.lower(): [v.lower() for v in values] for k, values in tech_map.items()}
    soft_map = {k.lower(): [v.lower() for v in values] for k, values in soft_map.items()}

    return tech_map, soft_map


# --------------------------
# 4. Extração + remoção → RETORNA macro + variações usadas
# --------------------------
def extract_skill_map(description: str, skill_map: dict):
    desc = description.lower()

    found = {}   # exemplo: {"python": ["python developer", "python3"]}

    for macro, variations in skill_map.items():
        matches = []
        for var in variations:
            if var in desc:
                matches.append(var)
                desc = desc.replace(var, "")  # remover do texto

        if matches:
            found[macro] = matches

    desc = re.sub(r"\s+", " ", desc).strip()
    return desc, found


# --------------------------
# 5. Processar cada job
# --------------------------
def process_job(job, tech_map, soft_map):

    desc = strip_html(job.get("description", "")).lower()

    # tech
    desc, tech_matches = extract_skill_map(desc, tech_map)

    # soft
    desc, soft_matches = extract_skill_map(desc, soft_map)

    job["description"] = desc
    job["tech_stacks_found"] = tech_matches
    job["soft_skills_found"] = soft_matches

    job.pop("tags", None)

    return job


# --------------------------
# 6. EXECUTAR PIPELINE E SALVAR
# --------------------------
def main():
    tech_map, soft_map = load_support_maps()

    with open(RAW_PATH, "r", encoding="utf-8") as f:
        raw_jobs = json.load(f)

    processed = [process_job(job, tech_map, soft_map) for job in raw_jobs]

    PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

    with open(PROCESSED_PATH, "w", encoding="utf-8") as f:
        json.dump(processed, f, indent=4, ensure_ascii=False)

    print(f"✔ Arquivo salvo em: {PROCESSED_PATH}")


if __name__ == "__main__":
    main()

✔ Arquivo salvo em: ../data/processed/jobs_processed.json


## Vetorizando campo description para usar como um dos parametros para nota a "proximidade" dos vetores

In [54]:
#load curriculo

curriculo = """
OBJECTIVE
Seeking career growth opportunities in the field of software development.

EDUCATION
Bachelor of Computer Science, Universidade Federal do ABC 
Expected Graduation: 2026

SKILLS
Main Technical Skills: Python, SQL, ETL, Java, Git
Soft Skills: Communication, Adaptability, Analytical Thinking, Proactivity
Github Portfolio: https://github.com/Netreck
Languages: English, Portuguese

PROFESSIONAL EXPERIENCE
Intern – Bank of America (Jun 2025 – Present)
VP Global Technology – Tech Rotation Program
• Developed test automation and tools to support QA teams in payment systems.
• Contributed to internal automation and process optimization projects.
• Technologies: Java, Python, Git, SQL, Node.js, HTML, CSS, Octane, QTest, Matera.

Data Intern – Vivo (Telefônica Brasil) (Jan 2025 – Jun 2025)
VP Engineering & Customer Services
• Generated, monitored, and analyzed operational KPIs focused on efficiency and customer experience.
• Built Machine Learning solutions to modernize and improve operational results.
• Technologies: Python, SQL, TensorFlow, Power BI, Excel.

EXTRACURRICULAR ACTIVITIES
Member – Green Team Hacker Club (Jan 2024 – Jan 2025)
Project Manager – Data Projects, Green Team Hacker Club (Jan 2025 – Present)
Federal University of ABC (UFABC) – Santo André, SP
• Official student-led organization linked to UFABC.
• Participated in classes and projects related to Data Science.
• Defined and managed technology stack for projects, considering scalability, performance, and integration.
• Managed team activities including task delegation, progress tracking, and technical support.
• Worked with ETL, process automation (pipelines), SQL, PostgreSQL, Data Visualization (Seaborn, Matplotlib), Data Modeling, Machine Learning models (Scikit-learn, TensorFlow), model deployment via APIs, MLflow, and LLMs.
"""

In [36]:
import pandas as pd

df = pd.read_json("../data/processed/jobs_processed.json")


In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 346 entries, 0 to 345
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   id                           346 non-null    int64 
 1   url                          346 non-null    object
 2   title                        346 non-null    object
 3   company_name                 346 non-null    object
 4   company_logo                 346 non-null    object
 5   category                     346 non-null    object
 6   job_type                     346 non-null    object
 7   publication_date             346 non-null    object
 8   candidate_required_location  346 non-null    object
 9   salary                       346 non-null    object
 10  description                  346 non-null    object
 11  tech_stacks_found            346 non-null    object
 12  soft_skills_found            346 non-null    object
 13  company_logo_url             9 non-

In [38]:
df.head()

,id,url,title,company_name,company_logo,category,job_type,publication_date,candidate_required_location,salary,description,tech_stacks_found,soft_skills_found,company_logo_url
0,2079700,https://remotive.com/remote-jobs/qa/junior-per...,Junior Performance QA Engineer,Veeam Software,https://remotive.com/job/2079700/logo,QA,full_time,2025-11-20T14:50:36,Poland,,"veeam, the #1 global maket leade in data esili...","{'languages': ['c', 'go', 'rust', 'r'], 'front...",{'creativity': ['ideation']},NaN
1,2079712,https://remotive.com/remote-jobs/software-deve...,Senior Backend Engineer,Roompricegenie,https://remotive.com/job/2079712/logo,Software Development,full_time,2025-11-20T10:50:04,Europe,,"about oompiegenie ✨🧞‍♂️founded in 2017, oompie...","{'languages': ['python', 'c', 'go', 'r'], 'fro...","{'creativity': ['innovation'], 'work_style': [...",NaN
2,2079711,https://remotive.com/remote-jobs/software-deve...,Senior Golang Engineer,Scale3c,https://remotive.com/job/2079711/logo,Software Development,full_time,2025-11-20T10:50:04,Europe,,join ou team to eshape dial logistis though ou...,"{'languages': ['c', 'go', 'r'], 'frontend_fram...","{'work_style': ['initiative', 'autonomy']}",NaN
3,2079698,https://remotive.com/remote-jobs/qa/senior-qa-...,Senior QA Automation Engineer,Cookunity,https://remotive.com/job/2079698/logo,QA,full_time,2025-11-20T08:50:39,Argentina,,about ook: food has lost its soul to moden onv...,"{'languages': ['java', 'typescript', 'c', 'go'...","{'creativity': ['ideation'], 'time_management'...",NaN
4,2079707,https://remotive.com/remote-jobs/software-deve...,iOS Developer,Just Play,https://remotive.com/job/2079707/logo,Software Development,full_time,2025-11-20T08:50:34,Germany,,justplay is looking fo an develope to play a k...,"{'languages': ['c', 'go', 'swift', 'r'], 'fron...",{},NaN


In [ ]:
### Tratando colunas
import ast

cols_list = [ "tech_stacks_found", "soft_skills_found"]

for col in cols_list:
    df[col] = df[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 346 entries, 0 to 345
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   id                           346 non-null    int64 
 1   url                          346 non-null    object
 2   title                        346 non-null    object
 3   company_name                 346 non-null    object
 4   company_logo                 346 non-null    object
 5   category                     346 non-null    object
 6   job_type                     346 non-null    object
 7   publication_date             346 non-null    object
 8   candidate_required_location  346 non-null    object
 9   salary                       346 non-null    object
 10  description                  346 non-null    object
 11  tech_stacks_found            346 non-null    object
 12  soft_skills_found            346 non-null    object
 13  company_logo_url             9 non-

In [ ]:
## Faz uma chamada por vaga para criar embbendingds das vagas

import pandas as pd

df["embedding"] = df["description"].apply(embeddings.embed_query)


In [44]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


In [55]:
### embendding curriculo
curriculo_emb = embeddings.embed_query(curriculo)

In [56]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# df_original é o seu df atual
df_original = df.copy()

# Calcula similaridade
df_original["embedding_similarity"] = df_original["embedding"].apply(
    lambda v: float(cosine_similarity(
        np.array(curriculo_emb).reshape(1, -1),
        np.array(v).reshape(1, -1)
    )[0][0])
)

# Ordena pela similaridade (desc)
df_sorted = df_original.sort_values(by="embedding_similarity", ascending=False)

df_sorted[["description", "tech_stacks_found", "embedding_similarity"]].head()

,description,tech_stacks_found,embedding_similarity
151,the ole: alphasights is seeking a highly expei...,"{'languages': ['python', 'java', 'typescript',...",0.435639
234,ompany desiptionweookit is an intenational sof...,"{'languages': ['python', 'java', 'c', 'go', 'r...",0.433487
276,addepto is a leading ai onsulting and data ene...,"{'languages': ['python', 'typescript', 'c', 'g...",0.423877
213,feedzai is the wold’s fist iskops platfom fo f...,"{'languages': ['python', 'c', 'go', 'rust', 'r...",0.415411
240,teavision tehnologies is a neashoe softwae out...,"{'languages': ['c', 'go', 'r', 'lua'], 'fronte...",0.413761


In [57]:
for idx, row in df_sorted.iterrows():
    print("="*70)
    print(f"📌 Job #{idx}")
    print(f"🏷️  Title: {row.get('title', 'N/A')}")
    print(f"🏢 Company: {row.get('company_name', 'N/A')}")
    print(f"🔧 Tech Stacks Found: {row.get('tech_stacks_found', [])}")
    print(f"📊 Similarity: {row['embedding_similarity']:.4f}")
    print("-"*70)
    print(f"📝 Description:\n{row.get('description','')[:500]}...")
    print("="*70)
    print()

📌 Job #151
🏷️  Title: Senior Quality Assurance Automation Engineer
🏢 Company: AlphaSights
🔧 Tech Stacks Found: {'languages': ['python', 'java', 'typescript', 'c', 'r'], 'frontend_frameworks': ['lit'], 'backend_frameworks': ['gin'], 'others_general': ['sql', 'automation', 'manual testing', 'testing', 'git', 'ios', 'api', 'unity']}
📊 Similarity: 0.4356
----------------------------------------------------------------------
📝 Description:
the ole: alphasights is seeking a highly expeiened, self-diven senio qa enee to join ou softwae eneeing team. we ae a dial business in whih ontinuous uptime, podut quay, and use expeiene ae itial to suess. the ole of the qa enee theefoe epesents a visible and valued oppot to have an immediate impat. woking alongside softwae eneeing and podut management, you will lead the design and implementation of automated solutions that sale aoss ou platfom, ensuing ou appliations ae eliable, salable, and me...

📌 Job #234
🏷️  Title: Senior Java Developer
🏢 Company: P

In [51]:
df_sorted.describe()

,id,embedding_similarity
count,3.460000e+02,346.000000
mean,2.061732e+06,0.357449
std,6.522461e+04,0.042062
min,1.359476e+06,0.189209
25%,2.067722e+06,0.332695
50%,2.070674e+06,0.360880
75%,2.072912e+06,0.385635
max,2.079712e+06,0.444239


In [58]:
output_path = "../data/processed/database.csv"
df_sorted.to_csv(output_path, index=False)

print(f"Arquivo salvo em: {output_path}")

Arquivo salvo em: ../data/processed/database.csv
